# Simulating self-excitation

Section 1.3 of the lecture notes, at the keyboard.  One two-dimensional Hawkes process, whose
types carry no meaning and are called type 1 and type 2, in three frozen parametrisations.  We
simulate it, count its events, and read what the branching matrix says about them and what it
does not.

Sections 1 to 3 go with §1.3.1 to §1.3.3, on counting processes, Hawkes processes and the
exponential kernel; section 4 goes with §1.3.4, on stability and clusters.

In [ ]:
from fractions import Fraction

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from unito26.lob import config
from unito26.lob.hawkes import Clock, ExponentialHawkes, HawkesParams
from unito26.lob.messages import BranchingRatio, Decay
from unito26.lob.visualization import (
    INK, MUTED, PALETTE, counting_figure, intensity_figure, raster_figure, use_template,
)

use_template()

SEED = 7
TYPES = ("type 1", "type 2")

## 1. A Poisson process, and the Fano factor

Example 1.3.1 builds a $d_E$-dimensional Poisson process from independent exponential waits:
the $j$-th arrival of type $e$ is the sum of $j$ waits of parameter $\lambda_e$.  We take two
types, at the mean rates of the first two parametrisations of section 2, so that this path
serves them as a Poisson control.  Merging the two sequences in time order gives the arrival
times $T_n$ of the ground process and the labels $E_n$, as in (1.3.2) and (1.3.3).

In [ ]:
HORIZON = 400_000.0
RATES = config.self_exciting_pair_params().stationary_intensity()


def poisson_path(rates, horizon, rng):
    '''Example 1.3.1 on (0, horizon]: the arrival times and the types of the ground process.'''
    times, types = [], []
    for e, rate in enumerate(rates):
        arrivals = np.cumsum(rng.exponential(1 / rate, int(2 * rate * horizon) + 10))
        assert arrivals[-1] > horizon
        arrivals = arrivals[arrivals <= horizon]
        times.append(arrivals)
        types.append(np.full(arrivals.size, e))
    times, types = np.concatenate(times), np.concatenate(types)
    order = np.argsort(times)
    return times[order], types[order]


poisson_times, poisson_types = poisson_path(RATES, HORIZON, np.random.default_rng(SEED))
print("rates (1/s)                     :", RATES)
print("events on (0, HORIZON]          :", f"{poisson_times.size:,}")
print("measured rate of each type (1/s):", np.round(np.bincount(poisson_types) / HORIZON, 4))

For a stationary counting process and a window of length $w$, the *Fano factor* is

$$F(w) = \frac{\operatorname{Var} N(w)}{\mathbb{E}\, N(w)} ,$$

with $N(w)$ the count in any window of that length.  In Example 1.3.1, $N_e(w)$ is Poisson of
parameter $\lambda_e w$ and the ground count is Poisson of parameter
$(\lambda_1 + \lambda_2) w$.  A Poisson variable has its variance equal to its mean, so
$F(w) = 1$ at every $w$, for each type and for the ground process.  A count more variable than
a Poisson count of the same mean has $F(w) > 1$, and section 3 shows that self-excitation
produces one.

We estimate $F(w)$ on one path, over its consecutive windows of length $w$: the sample
variance of their counts over the sample mean.

In [ ]:
MAX_WINDOWS = 1_000_000


def counts_in_windows(times, types, window):
    '''N_1 and N_2 in the consecutive windows [kw, (k+1)w) from time 0, a row per window:
    as many windows as fit in HORIZON, and at most MAX_WINDOWS.'''
    windows = min(int(HORIZON // window), MAX_WINDOWS)
    index = (times // window).astype(int)
    return np.column_stack([
        np.bincount(index[(types == e) & (index < windows)], minlength=windows)
        for e in (0, 1)
    ])


def fano(counts):
    '''Sample variance over sample mean, column by column.'''
    return counts.var(axis=0, ddof=1) / counts.mean(axis=0)


rows = []
for window in (0.1, 1.0, 10.0, 100.0):
    counts = counts_in_windows(poisson_times, poisson_types, window)
    rows.append({
        "w (s)": window,
        "windows": len(counts),
        "mean of N_g(w)": counts.sum(axis=1).mean(),
        "F, ground": fano(counts.sum(axis=1)),
        "F, type 1": fano(counts[:, 0]),
        "F, type 2": fano(counts[:, 1]),
        "corr(N_1, N_2)": np.corrcoef(counts.T)[0, 1],
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))

One at every window length, for the ground process and for each type, up to a sampling error
that grows as the windows become fewer.  The two counts are uncorrelated, the two types being
built from independent waits.  A Hawkes process departs from both.

## 2. A two-type Hawkes process

Definition 1.3.6 with the exponential kernel of Definition 1.3.9: a baseline $\mu \ge 0$, an
excitation matrix $A \ge 0$, and one decay $\beta$ shared by every pair of types.  The package
ships three such parametrisations of two types, frozen as records.

Each record is one entry $A_{e,e'}$, with `Component` the type excited, $e$, and `Cause` the
type exciting it, $e'$: row excited, column exciting, as in Remark 1.3.7.  The baseline sits on
the diagonal records, and the decay repeats.  The package counts types from 0, so component 0
is type 1.

In [ ]:
print(pd.DataFrame(config.ASYMMETRIC_PAIR_PARAMS).to_string(index=False))

In [ ]:
PAIRS = {
    "self-exciting": config.self_exciting_pair_params(),
    "cross-exciting": config.cross_exciting_pair_params(),
    "asymmetric": config.asymmetric_pair_params(),
}


def as_text(array):
    return str(np.round(array, 4).tolist())


print(pd.DataFrame({
    name: {
        "A (1/s)": as_text(params.excitation),
        "β (1/s)": as_text(params.decay),
        "Γ = A / β": as_text(params.branching_matrix),
        "eigenvalues of Γ": as_text(np.real_if_close(np.linalg.eigvals(params.branching_matrix))),
        "ρ": as_text(params.branching_ratio),
        "column sums of Γ": as_text(params.branching_matrix.sum(axis=0)),
        "column sums of A (1/s)": as_text(params.excitation.sum(axis=0)),
        "μ (1/s)": as_text(params.baseline),
        "λ* (1/s)": as_text(params.stationary_intensity()),
        "relaxation time (s)": as_text(params.relaxation_time()),
    }
    for name, params in PAIRS.items()
}).to_string())

$\Gamma = A/\beta$ (1.3.12) is dimensionless, while $A$ is a rate.  Both are read by columns.
By Remark 1.3.7 the column sums of $\Gamma$ are the expected numbers of direct offspring of one
event of each type; by Remark 1.3.10 the column sums of $A$ are the jumps in the total intensity
that the same event produces, measured per second.  The two differ by $\beta$.  The branching ratio
$\rho$ is the spectral radius of $\Gamma$, and $\lambda^* = (I - \Gamma)^{-1}\mu$ is the
stationary intensity of Theorem 1.3.19.

The self- and cross-exciting pairs share $\mu$, $\lambda^*$, $\rho$ and the column sums.  The
second is the first with its rows swapped, so that each type mostly excites the other rather
than itself.  In the asymmetric pair the column sums differ: an event of type 2 has more direct
offspring than an event of type 1, and most of them are of type 1.

The last row is the relaxation time of Definition 1.3.36, the time constant of the slowest mode
of the mean response; notebook 05 takes it up.

`ExponentialHawkes(params, seed).events(horizon)` yields the events $(T_n, E_n)$ of one path on
$(0, \mathrm{horizon}]$ in time order, the path started empty.  We use it as a black box here;
notebook 06 opens it.

A path started empty is not yet stationary.  We discard from every Hawkes path a warm-up of many
relaxation times, after which the empty start has left no appreciable trace, and read what
follows as, to a good approximation, the stationary process of Theorem 1.3.19.

In [ ]:
WARM_UP = 20.0


def simulate(params, horizon, seed):
    '''One path on (0, horizon], started empty: its event times and types.'''
    events = np.array(list(ExponentialHawkes(params, seed).events(horizon)))
    return events[:, 0], events[:, 1].astype(int)


def within(times, types, interval):
    '''The events of a path that fall in the interval (a, b].'''
    kept = (times > interval[0]) & (times <= interval[1])
    return times[kept], types[kept]


slowest = max(params.relaxation_time() for params in PAIRS.values())
print(f"warm-up of {WARM_UP:.0f} s: {WARM_UP / slowest:.0f} relaxation times of the slowest pair")

The same stretch of each pair on one seed, after the warm-up, beneath the same stretch of the
Poisson control of section 1.

In [ ]:
STRETCH = (WARM_UP, WARM_UP + 40.0)

paths = {"Poisson control": within(poisson_times, poisson_types, STRETCH)}
for name, params in PAIRS.items():
    paths[name] = within(*simulate(params, STRETCH[1], SEED), STRETCH)

figure = raster_figure(paths, TYPES, "The same stretch of each process, events by type")
figure.update_xaxes(range=STRETCH).update_yaxes(automargin=True)

The Poisson control has the mean rates of the first two pairs, and its events are scattered
with no tendency to bunch: their rate never depends on what came before.  The Hawkes events come
in bunches separated by quiet stretches, each bunch one cluster or a few, a cluster being an
immigrant and its descendants, as in Proposition 1.3.8.  In the self-exciting row the events of
a cluster mostly share one colour; in the cross-exciting row the two colours mix within a
cluster.  Those two rows have their events at the same times, which section 3 explains.  The
asymmetric row is denser, and mostly of type 1.

The same paths as counting processes.  By (1.3.4), $N_e(t)$ is the number of events of type
$e$ up to and including time $t$; it is drawn here counted from the start of the stretch, and
rises by one at each event of type $e$.  A bunch of the raster is a run of steps close
together, and a quiet stretch is a flat.

In [ ]:
counting_figure(
    paths, (WARM_UP, WARM_UP + 25.0), TYPES,
    "The first 25 s of the same stretch, events counted by type",
)

The intensity along the asymmetric path, from the start of the stretch above: `intensity_figure`
reads $\lambda = \mu + AZ$ (1.3.13) on a fine clock, and just before and just after every event.
Between events every component of $Z$ decays at the rate $\beta$; at an event of type $e'$, $Z$
gains the unit vector $\mathbf e_{e'}$ (1.3.14), so $\lambda$ jumps by the column
$A\mathbf e_{e'}$.

In [ ]:
asymmetric = PAIRS["asymmetric"]
zoom = (WARM_UP, WARM_UP + 15.0)
times, types = simulate(asymmetric, zoom[1], SEED)
intensity_figure(
    asymmetric, times, types, Clock(np.linspace(*zoom, 3001)), TYPES,
    "The asymmetric pair: the intensity of each type, and the events beneath",
)

At every event of type 2 the intensity of type 1 jumps by $A_{1,2}$ and that of type 2 by the
much smaller $A_{2,2}$; at an event of type 1 both jump by little.  A kernel transposed by
mistake would put the large jumps on type 2, at the events of type 1, and would pass every
stability test, since $\Gamma$ and $\Gamma^\top$ share their spectral radius (Remark 1.3.7).

## 3. Who triggers whom

The self- and cross-exciting pairs have the same $\mu$, $\lambda^*$ and $\rho$, and the column
sums of their $A$ all take one value, $c$.  Summing (1.3.13) over the types gives the intensity
of the ground process,

$$\lambda_{\mathfrak g}(t) = \bar\mu + \mathbf 1^\top A\, Z(t) = \bar\mu + c\,\mathbf 1^\top Z(t),
\qquad \mathbf 1^\top Z(t) = \sum_{n :\, T_n < t} e^{-\beta (t - T_n)} ,$$

a function of the past event times alone, and the same function for both pairs.  By
Proposition 1.3.3 the law of each waiting time given the past is then the same for both, and
depends on the past times alone; so the event times of the two pairs have the same law, that of
the one-dimensional Hawkes process with baseline $\bar\mu$, excitation $c$ and decay $\beta$.
Remark 1.3.45 explains why this fails in general: the jump of $\lambda_{\mathfrak g}$ at an event
is the column sum of $A$ for the type that fires, and it is the same for every type exactly
when the column sums are equal.

The simulator draws each waiting time in this way, from $\lambda_{\mathfrak g}$ alone, and then
draws the type, with probabilities $\lambda_e / \lambda_{\mathfrak g}$.  So on one seed the two
pairs produce the same event times and differ only in the types, and the one-dimensional
process with baseline $\bar\mu$, excitation $c$ and decay $\beta$ produces the same times too.
The raster of section 2 showed the first on a stretch; here are all three on a long path.

In [ ]:
def long_path(params):
    '''HORIZON seconds of one path on the notebook's seed, after the warm-up.'''
    times, types = simulate(params, WARM_UP + HORIZON, SEED)
    times, types = within(times, types, (WARM_UP, WARM_UP + HORIZON))
    return times - WARM_UP, types


self_exciting, cross_exciting = PAIRS["self-exciting"], PAIRS["cross-exciting"]
one_dimensional = HawkesParams(
    baseline=[self_exciting.baseline.sum()],
    excitation=[[self_exciting.excitation.sum(axis=0)[0]]],
    decay=self_exciting.decay,
)

self_times, self_types = long_path(self_exciting)
cross_times, cross_types = long_path(cross_exciting)
one_dimensional_times, _ = long_path(one_dimensional)

print(f"events in the self-exciting path: {self_times.size:,}")
for name, times in (("cross-exciting", cross_times), ("one-dimensional", one_dimensional_times)):
    assert times.size == self_times.size
    gap = np.abs(times - self_times).max()
    assert gap < 1e-9
    print(f"largest difference from its event times, {name:<15}: {gap:.1e} s")
print(f"events whose type differs, self- against cross-exciting: "
      f"{np.mean(self_types != cross_types):.1%}")

The times agree to rounding error, the three runs accumulating $\mathbf 1^\top Z$ from different
components.  The types of the two pairs differ on a large share of the events, and the two
pairs part in the statistics of the types.

The Fano factor of section 1, now on these two paths, in windows from much shorter to much
longer than the relaxation time, with the Poisson control beside them.  Consecutive windows of
a Hawkes path are not independent, but the process is ergodic (§1.3.4.4), so the sample
variance over the windows of one long path still estimates $\operatorname{Var} N(w)$.  Both
pairs are unchanged when the two types are swapped, so the two types have the same Fano factor,
and we report the mean of their two estimates.

In [ ]:
WINDOWS = np.geomspace(0.01, 100.0, 25)
PATHS = {
    "Poisson control": (poisson_times, poisson_types),
    "self-exciting": (self_times, self_types),
    "cross-exciting": (cross_times, cross_types),
}


def window_statistics(times, types, window):
    counts = counts_in_windows(times, types, window)
    return {
        "F, ground": fano(counts.sum(axis=1)),
        "F, per type": fano(counts).mean(),
        "corr(N_1, N_2)": np.corrcoef(counts.T)[0, 1],
    }


statistics = {
    name: pd.DataFrame([window_statistics(times, types, w) for w in WINDOWS], index=WINDOWS)
    for name, (times, types) in PATHS.items()
}
by_decade = pd.concat({name: frame.iloc[::6] for name, frame in statistics.items()}, axis=1)
by_decade.index = [f"w = {w:g} s" for w in WINDOWS[::6]]
print(by_decade.round(3).to_string())

In [ ]:
figure = go.Figure()
for name, column, label, colour, dash in (
    ("Poisson control", "F, ground", "Poisson control", MUTED, "dash"),
    ("self-exciting", "F, ground", "ground process, both pairs", INK, "solid"),
    ("self-exciting", "F, per type", "per type, self-exciting", PALETTE[2], "solid"),
    ("cross-exciting", "F, per type", "per type, cross-exciting", PALETTE[3], "solid"),
):
    figure.add_trace(go.Scatter(
        x=WINDOWS, y=statistics[name][column], name=label, mode="lines+markers",
        line=dict(color=colour, width=2, dash=dash), marker=dict(size=8),
    ))
figure.update_xaxes(type="log", title_text="window w (s)")
figure.update_yaxes(title_text="Fano factor F(w)", rangemode="tozero")
figure.update_layout(
    title="Fano factor against the length of the window", height=460,
    legend=dict(orientation="h", y=1.12, x=0),
)
figure

The Fano factors of the two ground processes coincide, window by window, since their times do.
They rise from one, at windows so short that a window rarely holds more than one event, and
level off at windows long against the relaxation time, where most clusters fall inside a single
window.  The Poisson control stays at one throughout.

Per type the pairs part.  Under self-excitation a cluster piles its events mostly on one type,
so each type's count is more dispersed than under cross-excitation, where a cluster spreads over
both types; and there the two counts move together, with a correlation close to one.

The order of the types along the ground process shows the same difference.  For the events of
each type, the shares of the two types among the events that immediately follow them:

In [ ]:
def transitions(types):
    '''Share of each next type (columns) after each type (rows), along the ground process.'''
    return pd.crosstab(
        types[:-1] + 1, types[1:] + 1,
        rownames=["after type"], colnames=["next type"], normalize="index",
    )


print(pd.concat(
    {"self-exciting": transitions(self_types), "cross-exciting": transitions(cross_types)},
    axis=1,
).round(3).to_string())

After an event of type 1 the self-exciting path goes on with type 1 more often than not, and
the cross-exciting path switches to type 2 more often than not.  Neither is close to certain:
the next event need not be a child of the last one, and may be an immigrant, a sibling of the
last one and most likely of its type, or a member of another cluster.

The two pairs have the same branching ratio and ground processes with the same law.  The types
record who triggers whom, and the branching ratio does not see it.

## 4. Reading the branching ratio

The branching ratio $\rho$ is the spectral radius of $\Gamma$.  The clusters of
Proposition 1.3.8 are described by two other numbers: the endogenous fraction
$1 - \bar\mu/\bar\lambda$ of Definition 1.3.22, the long-run share of events that are offspring
rather than immigrants, and the mean cluster size $\bar C = \bar\lambda/\bar\mu$ of
Proposition 1.3.23, which counts the immigrant; the first is $1 - 1/\bar C$.  The cluster
founded by an immigrant of type $e'$ has expected size $C_{e'}$, the $e'$-th entry of
$C = \mathbf 1^\top (I - \Gamma)^{-1}$ (1.3.22).  Remark 1.3.24 is the cheapest check of all
this: $\bar\mu\,\bar C = \bar\lambda$, since every event belongs to exactly one cluster.

Summing the fixed point $\lambda^* = \mu + \Gamma\lambda^*$ of Theorem 1.3.19 over the types
gives $\bar\lambda = \bar\mu + \mathbf 1^\top \Gamma \lambda^*$, so that

$$1 - \frac{\bar\mu}{\bar\lambda}
= \sum_{e'} \frac{\lambda^*_{e'}}{\bar\lambda} \, (\mathbf 1^\top \Gamma)_{e'} .$$

The endogenous fraction is the mean of the column sums of $\Gamma$, each type weighted by its
share of the events: the expected number of direct offspring of an event drawn at random from
the flow.

In [ ]:
def cluster_size_by_founder(params):
    '''C = 1' (I - Γ)^{-1}, eq. (1.3.22): the expected cluster size by the founder's type.'''
    identity = np.eye(params.dimension)
    return np.ones(params.dimension) @ np.linalg.inv(identity - params.branching_matrix)


def weighted_column_sum(params):
    '''The column sums of Γ, averaged over the types with the weights λ*_e' / λ̄.'''
    stationary = params.stationary_intensity()
    return params.branching_matrix.sum(axis=0) @ stationary / stationary.sum()


for params in PAIRS.values():
    assert np.isclose(
        params.baseline.sum() * params.mean_cluster_size(),
        params.stationary_intensity().sum(),
    )

print(pd.DataFrame({
    name: {
        "ρ": params.branching_ratio,
        "endogenous fraction": params.endogenous_fraction(),
        "column sums of Γ, weighted by λ*": weighted_column_sum(params),
        "1 / (1 - ρ)": 1 / (1 - params.branching_ratio),
        "mean cluster size": params.mean_cluster_size(),
        "C_1": cluster_size_by_founder(params)[0],
        "C_2": cluster_size_by_founder(params)[1],
        "immigration rate × mean cluster size": (
            params.baseline.sum() * params.mean_cluster_size()
        ),
        "total event rate": params.stationary_intensity().sum(),
    }
    for name, params in PAIRS.items()
}).round(4).to_string())

On the first two pairs every column sum equals $\rho$, so their weighted mean is $\rho$ whatever
the weights, and $\bar C = 1/(1-\rho)$ by Corollary 1.3.25.  On the asymmetric pair the column
sums straddle $\rho$ and the weights decide: here the endogenous fraction exceeds $\rho$, and
$\bar C$ exceeds $1/(1-\rho)$.  The first generation of a cluster founded by type $e'$ has
expected size $(\mathbf 1^\top\Gamma)_{e'}$, and only in late generations does the expected size
shrink by about the factor $\rho$ from one to the next.  $C_{e'}$ counts them all, and a founder
of type 2, whose column sum exceeds one, founds a cluster much larger than $1/(1-\rho)$.

Proposition 1.3.8 also says how to count a cluster without any times: an event of type $e'$ has
a Poisson number of direct offspring of type $e$, of mean $\Gamma_{e,e'}$, independently of
every other event.  Counted by generation, a cluster is a multitype Galton–Watson process, and
we simulate it as one, the founders drawn with probabilities $\mu_{e'}/\bar\mu$, as immigrants
are.

In [ ]:
def cluster_sizes(branching, founders, rng):
    '''The size of the cluster of each founder, the founder included.

    Column i of `generation` counts, by type, the current generation of cluster i; the next
    generation is Poisson with mean `branching @ generation`, as in Proposition 1.3.8.
    '''
    generation = np.eye(branching.shape[0], dtype=int)[:, founders]
    sizes = np.ones(founders.size, dtype=int)
    while generation.any():
        generation = rng.poisson(branching @ generation)
        sizes += generation.sum(axis=0)
    return sizes


def compared(formula, sample):
    '''A formula against the mean of a sample, with two standard errors of that mean.'''
    return {
        "formula": formula,
        "Monte Carlo": sample.mean(),
        "2 SE": 2 * sample.std(ddof=1) / np.sqrt(sample.size),
    }


rng = np.random.default_rng(SEED)
baseline = asymmetric.baseline
founders = rng.choice(asymmetric.dimension, size=200_000, p=baseline / baseline.sum())
sizes = cluster_sizes(asymmetric.branching_matrix, founders, rng)

by_founder = cluster_size_by_founder(asymmetric)
print(pd.DataFrame({
    "C_1, founder of type 1": compared(by_founder[0], sizes[founders == 0]),
    "C_2, founder of type 2": compared(by_founder[1], sizes[founders == 1]),
    "mean cluster size, founder drawn by μ": compared(asymmetric.mean_cluster_size(), sizes),
}).T.round(4).to_string())
print()
print(f"endogenous fraction: formula {asymmetric.endogenous_fraction():.4f}, "
      f"share of the simulated events that are not founders {1 - founders.size / sizes.sum():.4f}")

Proposition 1.3.23 holds within two standard errors, founder by founder and on average, and the
share of the simulated events that are not founders agrees with the endogenous fraction.

Now hold $\lambda^*$ and the shape of $A$, and move $\rho$.
`HawkesParams.from_stationary_intensity` scales the shape to the requested $\rho$ and solves
$\mu = (I - \Gamma)\lambda^*$ for the baseline.  The weights $\lambda^*_{e'}/\bar\lambda$ then
stay put while $\Gamma$ grows in proportion to $\rho$, so the endogenous fraction is a line
through the origin.

In [ ]:
stationary = asymmetric.stationary_intensity()
ladder = []
for rho in np.linspace(0.10, 0.65, 12):
    rung = HawkesParams.from_stationary_intensity(
        stationary, asymmetric.excitation, Decay(asymmetric.decay), BranchingRatio(rho)
    )
    assert np.allclose(rung.stationary_intensity(), stationary)
    ladder.append({
        "ρ": rung.branching_ratio,
        "μ_1": rung.baseline[0],
        "μ_2": rung.baseline[1],
        "endogenous fraction": rung.endogenous_fraction(),
        "1 / (1 - ρ)": 1 / (1 - rung.branching_ratio),
        "mean cluster size": rung.mean_cluster_size(),
    })
ladder = pd.DataFrame(ladder)

slope = weighted_column_sum(asymmetric) / asymmetric.branching_ratio
assert np.allclose(ladder["endogenous fraction"], slope * ladder["ρ"])

print(ladder.round(4).to_string(index=False))
print()
print(f"endogenous fraction = {slope:.4f} ρ = {Fraction(slope).limit_denominator(100)} ρ")

In [ ]:
figure = make_subplots(rows=1, cols=2, horizontal_spacing=0.12, subplot_titles=(
    "endogenous fraction against ρ", "mean cluster size against 1 / (1 - ρ)",
))
for column, (x, y) in enumerate(
    (("ρ", "endogenous fraction"), ("1 / (1 - ρ)", "mean cluster size")), start=1
):
    figure.add_trace(go.Scatter(
        x=ladder[x], y=ladder[y], mode="lines+markers", name="asymmetric shape at fixed λ*",
        line=dict(color=PALETTE[0], width=2), marker=dict(size=8),
        legendgroup="ladder", showlegend=column == 1,
    ), row=1, col=column)
    ends = [ladder[x].min(), ladder[x].max()]
    figure.add_trace(go.Scatter(
        x=ends, y=ends, mode="lines", name="equality",
        line=dict(color=MUTED, width=2, dash="dot"),
        legendgroup="equality", showlegend=column == 1,
    ), row=1, col=column)
    figure.update_xaxes(title_text=x, row=1, col=column)
    figure.update_yaxes(title_text=y, row=1, col=column)
figure.update_layout(height=440, legend=dict(orientation="h", y=1.2, x=0))
figure

The endogenous fraction lies on a line above the diagonal, and the mean cluster size above
$1/(1-\rho)$, and the gap widens as $\rho$ grows.  On this pair, reading either off $\rho$
understates how much of the flow the flow itself produces.

One rung further, the constructor refuses.

In [ ]:
try:
    HawkesParams.from_stationary_intensity(
        stationary, asymmetric.excitation, Decay(asymmetric.decay), BranchingRatio(0.7)
    )
except ValueError as error:
    print(error)

Component 0 is type 1.  On this shape $(\Gamma\lambda^*)_1 = 3\rho$, so
$\mu_1 = 2 - 3\rho$, which is negative above $\rho = 2/3$: no baseline sustains this
$\lambda^*$ there, baselines being non-negative by Definition 1.3.6.  A ladder at fixed
$\lambda^*$ ends where a component of $\mu$ reaches zero.

## What to take away

A Poisson count has a Fano factor of one in every window.  A Hawkes count is overdispersed, the
more so the longer the window, until most clusters fall inside a single window.

The branching ratio does not say who triggers whom.  The self- and cross-exciting pairs share
it, and their ground processes have the same law; they differ only in the statistics of the
types.

The branching ratio is a spectral radius, and the endogenous fraction is the mean of the column
sums of $\Gamma$ weighted by the stationary intensity.  The two agree in particular when the
column sums are constant, and in general they do not.